# Looking at chord annotations

What the Isophonics reference annotations contain, and how a backend's estimates compare
against them. Every plot here is a thin layer over `songscribe.eval.viz`, which does the
shaping and is unit-tested — so if a picture looks wrong, the bug is likely in the plotting
cell, not the data.

**Setup**

```
pip install -e '.[dev]'
jupyter lab notebooks/annotations.ipynb
```

Annotations are read from `SONGSCRIBE_EVAL_ANNOTATIONS` (default `data/isophonics`), which is
gitignored — see `EVALUATION.md` for how to obtain them. Nothing here needs audio or model
weights; the estimate cells need a `--cache` directory from a previous `songscribe-eval score`
run, and say so if it is missing.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from songscribe import lab
from songscribe.eval import dataset, metrics, viz

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MANIFEST = REPO / "eval/isophonics-beatles.json"
# Both default to the main checkout's gitignored data/. From a git worktree,
# where data/ lives in the main checkout rather than beside the code, set the
# environment variables instead.
ANNOTATIONS = Path(os.environ.get("SONGSCRIBE_EVAL_ANNOTATIONS", REPO / "data/isophonics"))
# From `songscribe-eval score ... --cache <dir>`; optional.
CACHE = Path(os.environ.get("SONGSCRIBE_EVAL_CACHE", REPO / "data/cache-aligned"))
BACKEND = "madmom"

plt.rcParams.update({"figure.figsize": (13, 3), "axes.grid": True, "grid.alpha": 0.25})

corpus = dataset.load_manifest(MANIFEST, annotation_root=ANNOTATIONS)
print(f"{len(corpus.tracks)} tracks, {len(corpus.missing_references())} missing annotations")

In [ ]:
# Pick a track. Any index works; this one is short and harmonically simple.
TRACK = next(t for t in corpus.tracks if "Eleanor_Rigby" in t.id)
reference = lab.read(TRACK.reference)
print(TRACK.id)
print(f"{len(reference)} annotated spans, {metrics.extent(reference)[1]:.1f}s")

## 1. The raw file

A `.lab` file is one chord per line: `start end label`, seconds from the beginning. Labels are
[Harte notation](https://ismir2005.ismir.net/proceedings/1080.pdf) — a bare root is a major
triad, `:min` a minor one, `/3` an inversion, `(9)` an added degree, `N` no chord sounding.

In [ ]:
for line in TRACK.reference.read_text().splitlines()[:10]:
    print(line)

## 2. The chord timeline

Each span drawn as a block, coloured by **root** — so the same root is the same colour
whatever its quality, and the harmonic motion is visible as a colour pattern.

The lower lane is the same data collapsed the way `render.py` would print it: repeats merged,
flicker under `align.MIN_CHORD_DURATION` dropped, `N` removed. The difference between the two
lanes is what the chart throws away.

In [ ]:
PALETTE = plt.get_cmap("twilight")


def draw_spans(ax, spans, label_every=True):
    """Draw chord spans as coloured blocks on one lane."""
    for span in spans:
        colour = "0.85" if span.colour == viz.NO_COLOUR else PALETTE(span.colour / 12)
        ax.barh(
            0,
            span.duration,
            left=span.start,
            height=1.0,
            color=colour,
            edgecolor="white",
            linewidth=0.7,
        )
        if label_every and span.duration > 1.5:
            ax.text(
                span.start + span.duration / 2,
                0,
                span.label,
                ha="center",
                va="center",
                fontsize=7.5,
                rotation=0,
            )
    ax.set_yticks([])
    ax.set_xlim(0, max((s.end for s in spans), default=1))


fig, axes = plt.subplots(2, 1, figsize=(13, 3.2), sharex=True)
draw_spans(axes[0], viz.chord_spans(reference))
axes[0].set_ylabel("raw", rotation=0, ha="right", va="center")
draw_spans(axes[1], viz.chord_spans(reference, vocabulary="majmin", chart=True))
axes[1].set_ylabel("chart", rotation=0, ha="right", va="center")
axes[1].set_xlabel("seconds")
fig.suptitle(TRACK.id.split("/")[-1], y=1.02)
plt.tight_layout()

## 3. Harmonic rhythm

How long chords are held. A regular harmonic rhythm shows up as peaks at multiples of the
bar; a broad smear means the chords do not fall on a grid — which is exactly when
`metrics.grid_score` refuses to report a number.

In [ ]:
chart_events = metrics.dedupe_chart(reference, "majmin")
bins = viz.duration_bins(chart_events, width=0.25, limit=10.0)
rhythm = metrics.harmonic_rhythm(chart_events)

fig, ax = plt.subplots(figsize=(13, 2.8))
ax.bar([b[0] for b in bins], [b[1] for b in bins], width=0.22, color="#4C72B0")
ax.axvline(
    rhythm.unit,
    color="crimson",
    ls="--",
    label=f"estimated unit {rhythm.unit:.2f}s (residual {rhythm.residual:.1%})",
)
ax.set_xlabel("chord duration (s)")
ax.set_ylabel("chords")
ax.legend()
ax.set_title(f"harmonic rhythm — grid is {'usable' if rhythm.is_reliable else 'NOT recoverable'}")
plt.tight_layout()

## 4. Which roots the track lives in

Seconds per root, every root shown. The gaps are the informative part: this reads as a key
profile rather than an arbitrary bar chart.

In [ ]:
roots = viz.root_time(reference)
fig, ax = plt.subplots(figsize=(13, 2.6))
ax.bar([r[0] for r in roots], [r[1] for r in roots], color=[PALETTE(i / 12) for i in range(12)])
ax.set_ylabel("seconds")
ax.set_title("time per chord root")
plt.tight_layout()

## 5. The corpus as a whole

Label coverage across all 180 tracks. The head is a handful of plain triads; the tail is long
and thin — 407 distinct labels, most accounting for seconds. `songscribe-eval survey` reports
the same thing numerically, including how much of it each comparison vocabulary can score.

In [ ]:
everything = [e for t in corpus.tracks for e in lab.read(t.reference)]
top = viz.label_time(everything, top=25)
total = sum(seconds for _, seconds in viz.label_time(everything))

fig, ax = plt.subplots(figsize=(13, 3.4))
ax.bar([t[0] for t in top], [t[1] / 60 for t in top], color="#55A868")
ax.set_ylabel("minutes")
ax.set_title(
    f"25 most-used labels of {len(viz.label_time(everything))}"
    f" — {sum(t[1] for t in top) / total:.0%} of {total / 3600:.1f}h"
)
plt.xticks(rotation=60, ha="right")
plt.tight_layout()

## 6. Reference against an estimate

Needs a `--cache` from a previous scoring run. The third lane is the agreement strip: green
where the estimate is right, red where it is wrong, grey where the reference is not scoreable
in this vocabulary. A run of red is a passage the model lost — far more legible than the
single WCSR number it averages into.

In [ ]:
estimate_path = CACHE / BACKEND / f"{TRACK.id.replace('/', '__')}.lab"
if not estimate_path.exists():
    print(f"no cached estimate at {estimate_path}")
    print(f"run: songscribe-eval score {MANIFEST} -c {BACKEND} --cache {CACHE}")
else:
    estimate = lab.read(estimate_path)
    fig, axes = plt.subplots(
        3, 1, figsize=(13, 4.0), sharex=True, gridspec_kw={"height_ratios": [3, 3, 1]}
    )
    draw_spans(axes[0], viz.chord_spans(reference, vocabulary="majmin", chart=True))
    axes[0].set_ylabel("reference", rotation=0, ha="right", va="center")
    draw_spans(axes[1], viz.chord_spans(estimate, vocabulary="majmin", chart=True))
    axes[1].set_ylabel(BACKEND, rotation=0, ha="right", va="center")

    for low, high, verdict in viz.agreement_spans(reference, estimate):
        colour = {True: "#55A868", False: "#C44E52", None: "0.8"}[verdict]
        axes[2].barh(0, high - low, left=low, height=1.0, color=colour)
    axes[2].set_yticks([])
    axes[2].set_ylabel("agree", rotation=0, ha="right", va="center")
    axes[2].set_xlabel("seconds")
    axes[2].legend(
        handles=[
            Patch(color="#55A868", label="correct"),
            Patch(color="#C44E52", label="wrong"),
            Patch(color="0.8", label="not scoreable"),
        ],
        ncol=3,
        loc="upper right",
        fontsize=8,
    )

    print(f"majmin    {metrics.score(reference, estimate, 'majmin').wcsr:.2%}")
    print(f"chart     {metrics.chart_score(reference, estimate):.2%}")
    print(f"grid      {metrics.grid_score(reference, estimate):.2%}")
    plt.tight_layout()

## 7. Is the recording on the reference's timeline?

WCSR as a function of a global time shift. A clean peak **at zero** means the audio and the
annotation share a clock. A peak elsewhere means an encoder lead-in or the wrong master, and
the headline score is measuring that mismatch rather than the model — this is the check
`songscribe-eval score` runs automatically.

Never read the peak as a corrected score: picking the offset that scores best is fitting to
the reference. Trim the audio instead and re-run with a fresh cache.

In [ ]:
if estimate_path.exists():
    curve = viz.offset_curve(reference, estimate, search=3.0, step=0.05)
    best = max(curve, key=lambda point: point[1])

    fig, ax = plt.subplots(figsize=(13, 3.0))
    ax.plot([c[0] for c in curve], [c[1] * 100 for c in curve], color="#4C72B0")
    ax.axvline(0, color="0.5", lw=1, label="no shift")
    ax.plot(
        best[0], best[1] * 100, "o", color="crimson", label=f"best {best[0]:+.2f}s at {best[1]:.1%}"
    )
    ax.set_xlabel("global shift applied to the estimate (s)")
    ax.set_ylabel("majmin WCSR (%)")
    ax.legend()
    ax.set_title("aligned" if abs(best[0]) < 0.15 else "MISALIGNED — fix the audio")
    plt.tight_layout()